# 03 · An agent with authenticated MCP servers

Two kinds of credentials, the way Claude's connectors work:
- **API key** on your own server (`app/shop_server_auth.py`): every request sends `Authorization: Bearer <key>`;
  the read key (scope `shop:read`) sees the 3 read tools, the write key (`shop:read shop:write`) also sees
  `update_order_status`. No key: HTTP 401.
- **OAuth sign-in** on a third-party server (Notion): the first connection opens the Notion sign-in page and waits;
  the token is stored encrypted in `~/.cole-agent/oauth-tokens/` and reused next time.

**Before you run this notebook**, start the server in a terminal opened in the session folder (keep it running):
```bash
uv run app/shop_server_auth.py
```
The server terminal logs its tool calls and refusals; its audit log is `output/audit.jsonl`.
The connection code is the one of the [LangChain MCP authentication docs](https://docs.langchain.com/oss/python/langchain/mcp/auth):
one FastMCP `Client` per server, each with its own credential, in a `ClientGroup`. Steps and other MCP servers:
`docs/03_auth.md`.

<img src="images/b3_scopes.png" width="640" alt="the key decides which tools exist">

<img src="images/b3_oauth_discovery.png" width="640" alt="OAuth sign-in: find the authorization server, get a client_id">

<img src="images/b3_oauth_login.png" width="640" alt="OAuth sign-in: PKCE login, token, MCP call">

Docs: [MCP authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization) ·
[FastMCP OAuth client](https://gofastmcp.com/clients/auth/oauth)

In [1]:
import json
import logging
import os
import re
import time
import warnings
from pathlib import Path

import httpx
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core._api import LangChainBetaWarning

os.environ["FASTMCP_LOG_ENABLED"] = "false"  # before fastmcp is imported: no rich handler of its own,
                                             # its log lines go through the format below
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2", "mcp", "fastmcp"):  # HTTP and MCP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logging.getLogger("fastmcp.client.auth").setLevel(logging.INFO)  # keeps FastMCP's "OAuth authorization URL: ..." line
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=LangChainBetaWarning)  # langchain.mcp is in beta

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY", "SHOP_READ_KEY", "SHOP_WRITE_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
if not os.getenv("OAUTH_STORAGE_ENCRYPTION_KEY", "").strip():  # encrypts the stored Notion sign-in (section 3)
    raise RuntimeError("Missing OAUTH_STORAGE_ENCRYPTION_KEY in .env. Make one in a terminal (session folder) with\n"
                       "  uv run python -c \"from cryptography.fernet import Fernet; print(Fernet.generate_key().decode())\"\n"
                       "paste it into .env, then restart the kernel.")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

# SHOP_AUTH_MCP_URL (optional): the --public URL; by default the local server on SHOP_AUTH_PORT
SHOP_URL = os.getenv("SHOP_AUTH_MCP_URL") or f"http://localhost:{os.getenv('SHOP_AUTH_PORT', '8001')}/mcp"


def check_server(url: str, start: str) -> None:
    """Fail fast with the command to run when the MCP server is not up."""
    try:
        httpx.get(url.removesuffix("/mcp") + "/healthz", timeout=3).raise_for_status()
    except httpx.HTTPError:
        raise RuntimeError(f"No MCP server at {url}. Start it in a terminal (session folder): {start}") from None
    logger.info("shop server: ok")  # no URL in the output: it depends on this machine's port


check_server(SHOP_URL, "uv run app/shop_server_auth.py")

INFO | Model: gpt-6-luna
INFO | shop server: ok


## 1. Local tools

In [2]:
# ── Local tools: web search and page reading (copied from Session 02) ───────
from langchain.tools import tool
from tavily import BadRequestError, TavilyClient

tavily = TavilyClient()  # uses TAVILY_API_KEY


def log_call(tool: str, args: str, status: str, started: float) -> None:
    """The one log line of a tool call: tool(short args) -> ok|error <ms> ms."""
    args = " ".join(args.split())
    logger.info("%s(%s) -> %s %d ms", tool, args if len(args) <= 70 else args[:70] + "…", status,
                (time.monotonic() - started) * 1000)


@tool
def search_docs(query: str) -> list[dict]:
    """Search the web for documents about the query. Returns the title and url of the top 3 results."""
    started, status = time.monotonic(), "error"
    try:
        hits = tavily.search(query, max_results=3)["results"]
        status = "ok"
        return [{"title": h["title"], "url": h["url"]} for h in hits]
    finally:
        log_call("search_docs", repr(query), status, started)


@tool
def read_doc(url: str, question: str) -> str:
    """Read the parts of one document (a web page from search_docs) that answer the question."""
    started, status = time.monotonic(), "error"
    try:
        pages = tavily.extract(urls=[url], query=question, chunks_per_source=3)["results"]
        if not pages:
            return f"ERROR: could not read {url}"
        status = "ok"
        return pages[0]["raw_content"]
    except BadRequestError as error:  # bad input: tell the model so it can pick another page
        return f"ERROR: could not read {url} ({error})"
    finally:
        log_call("read_doc", repr(url), status, started)


## 2. Your server with an API key
Without a key the server answers **401** before any tool runs. With the read key the write tool is not even listed,
so the model cannot choose it; calling it anyway gives `Unknown tool`.

In [3]:
from fastmcp import Client
from fastmcp.exceptions import ToolError

r = httpx.post(SHOP_URL, json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
               headers={"Accept": "application/json, text/event-stream"})
scope = re.search(r'scope="([^"]+)"', r.headers.get("www-authenticate", ""))
logger.info("no key    -> HTTP %s, required scope %s", r.status_code, scope and scope[1])

# auth="<key>" sends the header Authorization: Bearer <key> (FastMCP bearer auth)
async with Client(SHOP_URL, auth=os.environ["SHOP_READ_KEY"]) as c:
    logger.info("read key  -> %s", [t.name for t in await c.list_tools()])
    try:
        await c.call_tool("update_order_status", {"order_id": 1500, "status": "shipped"})
    except ToolError as e:
        logger.warning("read key calls update_order_status -> %s", e)

async with Client(SHOP_URL, auth=os.environ["SHOP_WRITE_KEY"]) as c:
    logger.info("write key -> %s", [t.name for t in await c.list_tools()])

INFO | no key    -> HTTP 401, required scope shop:read
INFO | read key  -> ['list_tables', 'describe_table', 'run_query']
WARNING | read key calls update_order_status -> Unknown tool: 'update_order_status'
INFO | write key -> ['list_tables', 'describe_table', 'run_query', 'update_order_status']


## 3. MCP tools: your server (API key) and Notion (OAuth)
Exactly the [per-server authentication](https://docs.langchain.com/oss/python/langchain/mcp/auth#per-server-authentication)
pattern of the LangChain docs: one FastMCP `Client` per server, each with its own `auth`, in a `ClientGroup` adapted by
`MCPAdapter`. Tool names get the group key as prefix (`shop_run_query`, `notion_notion-search`), so two servers never clash.

- `shop`: `auth="<key>"`, a bearer token (the read key).
- `notion`: FastMCP's `OAuth` helper runs the whole OAuth 2.1 flow: discovery, client registration, the browser sign-in
  (PKCE) and the token exchange. Its `token_storage` keeps the tokens on disk, encrypted, as in the
  [FastMCP OAuth docs](https://gofastmcp.com/clients/auth/oauth#token-storage); without it they live in memory and every
  run signs in again. When no valid token is stored, connecting opens the Notion sign-in page and **waits** (5 minutes)
  until you click **Allow**; later runs reuse the token, refreshed automatically.

Adding another server is one more `Client(url, auth=...)` entry in the group (`docs/03_auth.md`). The agent gets only 3 of
Notion's tools.

In [4]:
from cryptography.fernet import Fernet
from fastmcp.client.auth import OAuth
from fastmcp.client.group import ClientGroup
from key_value.aio.stores.disk import DiskStore
from key_value.aio.wrappers.encryption import FernetEncryptionWrapper
from langchain.mcp import MCPAdapter

# Tokens on disk, encrypted with your key from .env: the sign-in happens once, not on every run.
token_store = FernetEncryptionWrapper(
    key_value=DiskStore(directory=Path.home() / ".cole-agent" / "oauth-tokens"),
    fernet=Fernet(os.environ["OAUTH_STORAGE_ENCRYPTION_KEY"]),
)
notion_auth = OAuth(
    mcp_url="https://mcp.notion.com/mcp",
    token_storage=token_store,
    callback_port=8790,  # fixed redirect URI http://localhost:8790/callback
    client_name="COLE lab agent",
    # Register as a public client (PKCE only, no secret), like a desktop app:
    # Notion rejects the default secret-based registration at the token exchange.
    additional_client_metadata={"token_endpoint_auth_method": "none"},
)
servers = ClientGroup({
    "shop": Client(SHOP_URL, auth=os.environ["SHOP_READ_KEY"]),          # bearer token
    "notion": Client("https://mcp.notion.com/mcp", auth=notion_auth),     # OAuth 2.1 sign-in
})

print("Not signed in to Notion yet? The browser opens; the link is also logged as 'OAuth authorization URL'. "
      "Browser on another machine: after Allow, copy the localhost:8790/callback?... URL and run curl '<url>' here.")
async with MCPAdapter(servers) as mcp:
    tools = await mcp.list_tools()

NOTION_TOOLS = ["notion_notion-search", "notion_notion-fetch", "notion_notion-create-pages"]
mcp_tools = [t for t in tools if t.name.startswith("shop_") or t.name in NOTION_TOOLS]
logger.info("MCP tools: %s", [t.name for t in mcp_tools])

Not signed in to Notion yet? The browser opens; the link is also logged as 'OAuth authorization URL'. Browser on another machine: after Allow, copy the localhost:8790/callback?... URL and run curl '<url>' here.


INFO | MCP tools: ['shop_list_tables', 'shop_describe_table', 'shop_run_query', 'notion_notion-search', 'notion_notion-fetch', 'notion_notion-create-pages']


## 4. Agent
Local tools, your server (read key) and Notion in one list. `ToolRetryMiddleware` retries only transient failures
(timeouts, dropped connections); a refused call (401, `Unknown tool`) is not retried. Where to save in Notion is the
agent's decision, written in the prompt: a page link in the question, else a page it finds, else a new private page.

**Logging vs streaming.** *Logging* is diagnostics for the developer: lines your code and the servers write with
`logger.info` (`INFO | …`), kept or dropped by level. *Streaming* is the agent's own output delivered to the user while
it is produced: `stream()` uses `agent.astream(..., stream_mode=["updates", "messages"])`, prints one line per tool
call (`→`) and tool result (`←`) from `updates`, and the answer token by token from `messages`, instead of waiting
silently for the whole run. Docs: [Streaming](https://docs.langchain.com/oss/python/langchain/streaming)

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware, ToolRetryMiddleware
from langchain.messages import AIMessageChunk, ToolMessage
from langchain_core.tools import ToolException
from mcp.shared.exceptions import MCPError
from mcp_types import CONNECTION_CLOSED, REQUEST_TIMEOUT


def transient(exc: Exception) -> bool:
    """Retry timeouts and dropped connections only."""
    return (isinstance(exc, (TimeoutError, ConnectionError))
            or (isinstance(exc, MCPError) and exc.code in (CONNECTION_CLOSED, REQUEST_TIMEOUT)))


def on_error(exc: Exception, request) -> str | None:
    """After the retries, tell the model what failed instead of crashing the run."""
    if transient(exc) or isinstance(exc, ToolException):
        return f"{request.tool_call['name']} failed: {exc}. Tell the user; do not retry."


agent = create_agent(
    model,
    tools=[search_docs, read_doc, *mcp_tools],
    system_prompt=("You are an assistant for the owner of an online store. Amounts are in VND. Query the shop "
                   "database with one aggregate SQL query per question when possible; use the web only for outside "
                   "context and cite its url. To save to Notion: if the user gives a Notion page link, create the new "
                   "page under it; otherwise search Notion for the page the user names and create it there; if there "
                   "is none, create the new page as a private page (no parent). Give the url of the page you created."),
    middleware=[ToolErrorMiddleware(on_error),  # outer: turns the final failure into a ToolMessage
                ToolRetryMiddleware(max_retries=2, initial_delay=0.5, retry_on=transient, on_failure="error")],
)


def short(text: str, n: int = 110) -> str:
    return text if len(text) <= n else text[:n] + "…"


async def stream(agent, question: str) -> list:
    """Stream one run: a line per tool call and tool result ("updates"), the answer token by token ("messages").
    Returns the messages the run added."""
    added, mid_line = [], False
    async for chunk in agent.astream({"messages": [{"role": "user", "content": question}]},
                                     stream_mode=["updates", "messages"], version="v2"):
        if chunk["type"] == "messages":  # (token, metadata): print the model's text as it is generated
            token, meta = chunk["data"]
            if isinstance(token, AIMessageChunk) and meta["langgraph_node"] == "model" and token.text:
                print(token.text, end="", flush=True)
                mid_line = True
            continue
        for update in chunk["data"].values():  # updates: {node: what the node added to the state}
            for m in (update or {}).get("messages", []):
                added.append(m)
                lines = [f"→ {c['name']}({short(json.dumps(c['args'], ensure_ascii=False))})"
                         for c in getattr(m, "tool_calls", [])]
                if isinstance(m, ToolMessage):
                    lines.append(f"← {m.name} {m.status}: {short(m.text)}")
                if lines:
                    print(("\n" if mid_line else "") + "\n".join(lines), flush=True)
                    mid_line = False
    print()
    return added

## 5. Demo
The task: a Q3 sales analysis from the database and the web, saved as a new Notion page. Each run creates one page.

The demo is one turn, so the question is a plain string: edit it and run the cell again. Reading the user's input
(`input()`, a chat UI) is only needed when the agent talks back: multi-turn, resume, human-in-the-loop (Session 02).

To choose the place yourself, paste a Notion page link into the question (**Share → Copy link** in Notion).

In [6]:
question = ("Write a short sales analysis for Q3 2026 (July to September) of our online store: revenue by product "
            "category, the 3 cities with the most revenue, one web source about a trend for the top category, and one "
            "recommendation, under 200 words. Save it as a new Notion page in my page 'COLE Agent Reports'.")
messages = await stream(agent, question)

→ shop_list_tables({})
→ notion_notion-search({"query": "COLE Agent Reports", "query_type": "internal", "page_size": 10})
← shop_list_tables success: ["customers","order_items","orders","products"]
← notion_notion-search success: {"results":[{"id":"3eec87cb-a093-813a-86ed-cf3dc7aa4e77","title":"COLE Agent Reports","url":"https://app.notio…
→ shop_describe_table({"table": "orders"})
→ shop_describe_table({"table": "order_items"})
→ shop_describe_table({"table": "products"})
→ shop_describe_table({"table": "customers"})
← shop_describe_table success: {"table":"orders","columns":[{"name":"id","type":"INTEGER"},{"name":"customer_id","type":"INTEGER"},{"name":"s…
← shop_describe_table success: {"table":"products","columns":[{"name":"id","type":"INTEGER"},{"name":"name","type":"TEXT"},{"name":"category"…
← shop_describe_table success: {"table":"order_items","columns":[{"name":"order_id","type":"INTEGER"},{"name":"product_id","type":"INTEGER"},…
← shop_describe_table success: {"table":"custom

INFO | search_docs('2026 home furnishings trend compact urban furniture market trend') -> ok 2217 ms


← search_docs success: [{"title": "Furniture Trends 2026 Brand Should Watch Out For", "url": "https://www.vividworks.com/blog/furnitu…
→ read_doc({"url": "https://www.vividworks.com/blog/furniture-trends-2026", "question": "What 2026 furniture/home trend d…)


INFO | read_doc('https://www.vividworks.com/blog/furniture-trends-2026') -> ok 357 ms


← read_doc success: 1. Modular functionality

Modular functionality remains central to furniture trends for 2026, particularly in …
→ notion_notion-fetch({"id": "notion://docs/enhanced-markdown-spec"})
← notion_notion-fetch success: {"metadata":{"type":"documentation"},"title":"Enhanced Markdown Specification","url":"notion://docs/enhanced-m…
→ notion_notion-create-pages({"parent": {"page_id": "3eec87cb-a093-813a-86ed-cf3dc7aa4e77"}, "pages": [{"properties": {"title": "Q3 2026 Sa…)
← notion_notion-create-pages success: {"pages":[{"id":"3f0c87cb-a093-81b5-a93f-fcc87ee2d2af","url":"https://app.notion.com/p/3f0c87cba09381b5a93ffcc…
Created the Q3 2026 sales analysis under **COLE Agent Reports**. It includes category revenue, the top three cities, a linked 2026 home-furnishings trend, and a recommendation.

[Open the Notion page](https://app.notion.com/p/3f0c87cba09381b5a93ffcc87ee2d2af?pvs=204)


The server wrote one audit record per tool call it served (who, which tool, which arguments, the outcome):

In [7]:
audit = Path("../output/audit.jsonl")
for line in audit.read_text().splitlines()[-4:] if audit.exists() else []:
    print(line[:160])

{"ts": "2026-10-05T03:11:21+00:00", "client_id": "analyst", "tool": "describe_table", "args": {"table": "order_items"}, "status": "ok", "ms": 2}
{"ts": "2026-10-05T03:11:21+00:00", "client_id": "analyst", "tool": "describe_table", "args": {"table": "customers"}, "status": "ok", "ms": 3}
{"ts": "2026-10-05T03:11:27+00:00", "client_id": "analyst", "tool": "run_query", "args": {"sql": "SELECT o.status, COUNT(DISTINCT o.id) AS orders, SUM(oi.quanti
{"ts": "2026-10-05T03:11:32+00:00", "client_id": "analyst", "tool": "run_query", "args": {"sql": "WITH q3_sales AS (SELECT o.id AS order_id, o.customer_id, oi.p
